# A1 — Adaptive Autonomy

**Multi-agent DAG with state sharing, dependencies, and conditional execution.**

A1 builds on A0 by adding:
- **2+ agents** connected by explicit dependencies (`depends_on`)
- **State sharing** — agents pass data to downstream agents via `share_output`
- **Conditional execution** — branches, fan-out/fan-in, loops
- A **state sharing strategy** must be declared

### A0 vs A1 — The Key Difference

| | A0 | A1 |
|---|---|---|
| Agents | 1+ (isolated) | 2+ (connected) |
| Dependencies | None | Explicit `depends_on` edges |
| State sharing | Minimal | `share_output` fields flow downstream |
| Execution | Sequential, one at a time | Topological — parallel where possible |
| Use case | Single-task agent | Multi-step pipelines (plan → research → write) |

### When to use A1
- Multi-step pipelines with clear handoff points
- Tasks where each agent specializes (planner, researcher, writer)
- When you want predictable execution but need collaboration between agents

## 1. Provider Setup

In [ ]:
PROVIDER = "openrouter"
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OPENROUTER_API_KEY = ""
OPENROUTER_MODEL = "openai/gpt-5-mini"
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key or not url:
        raise ValueError("Set CUSTOM_API_KEY and CUSTOM_BASE_URL")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'")

os.environ["LLM_MODEL"] = MODEL
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## 2. The A1 Workflow — Multi-Agent DAG

The `02-research-pipeline` example has three agents in a linear chain:

```
  [planner]  ──────────────────►  [researcher]  ──────────────►  [writer]
     │                                  │                            │
     │ shares:                          │ shares:                    │ shares:
     │  • research_questions            │  • findings                │  • report
     │  • search_strategy               │  • sources                 │
     ▼                                  ▼                            ▼
  (state dict grows)              (state dict grows)           (final output)
```

### The YAML difference from A0

```yaml
orchestration:
  graph:
    - id: planner
      agent: planner
      depends_on: []                    # Root node
      share_output:                     # NEW at A1: explicit state sharing
        - research_questions
        - search_strategy

    - id: researcher
      agent: researcher
      depends_on: [planner]             # NEW at A1: dependency edge
      share_output:
        - findings
        - sources

    - id: writer
      agent: writer
      depends_on: [researcher]          # Runs only after researcher completes
      share_output:
        - report

state:
  model: shared_dict
  sharing:
    strategy: full                      # NEW at A1: sharing strategy required
```

**New at A1:**
1. Multiple agents with `depends_on` edges
2. `share_output` fields that flow data downstream
3. `state.sharing.strategy` declaration

## 3. Load and Validate the Multi-Agent DAG

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import validate_graph, check_compliance

PROJECT = Path("/home/shumway/projects/agent-workflow-protocol")
workflow_dir = PROJECT / "examples" / "02-research-pipeline"

manifest = parse_manifest(workflow_dir / "workflow.awp.yaml")

print(f"Workflow: {manifest.workflow.name}")
print(f"Tags:     {manifest.workflow.tags}")
print()

# Show graph structure with dependencies
print("DAG Structure:")
print("-" * 50)
for node in manifest.orchestration.graph:
    deps = node.depends_on or []
    shared = node.share_output or []
    print(f"  {node.id}")
    print(f"    depends_on:   {deps if deps else '(root)'}")
    print(f"    share_output: {shared}")
    print()

# Validate graph
graph_result = validate_graph(manifest.orchestration)
print(f"Graph valid: {graph_result.valid}")

## 4. State Sharing — The Core A1 Feature

State sharing is what makes A1 more than just "run agents in order".
Each agent receives the accumulated state from its predecessors and can contribute new fields.

In [ ]:
# Visualize the data flow through the pipeline
print("Data Flow Through the Pipeline")
print("=" * 60)

accumulated_state = ["task"]

for node in manifest.orchestration.graph:
    deps = node.depends_on or []
    shared = node.share_output or []

    print(f"\n>>> Agent '{node.id}' executes")
    print(f"    Receives state: {accumulated_state}")
    print(f"    Produces:       {shared}")

    accumulated_state.extend(shared)
    print(f"    State after:    {accumulated_state}")

print("\n" + "=" * 60)
print(f"Final state contains {len(accumulated_state)} fields: {accumulated_state}")

## 5. Execute the A1 Pipeline

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from awp.runtime import WorkflowRunner

runner = WorkflowRunner(workflow_dir)
result = runner.run("Research the current state of quantum computing and its practical applications in 2026")

print("\n" + "=" * 60)
print("A1 PIPELINE RESULT")
print("=" * 60)

for agent_name in ["planner", "researcher", "writer"]:
    agent_result = result.get(agent_name, {})
    confidence = agent_result.get("confidence", 0.0)
    has_error = "error" in agent_result
    icon = "X" if has_error else "OK"

    print(f"\n[{icon}] {agent_name} (confidence: {confidence})")
    for key, value in agent_result.items():
        if key == "confidence":
            continue
        text = str(value)
        if len(text) > 300:
            text = text[:300] + "..."
        print(f"    {key}: {text}")

## 6. Compare: A0 vs A1 Execution

Let's run the A0 hello-world side-by-side to highlight the structural difference.

In [ ]:
# A0: single agent, no state sharing
a0_dir = PROJECT / "examples" / "01-hello-world"
a0_manifest = parse_manifest(a0_dir / "workflow.awp.yaml")

print("A0 (Prescribed) — Hello World")
print(f"  Agents:        {[n.id for n in a0_manifest.orchestration.graph]}")
print(f"  Dependencies:  None")
print(f"  State sharing: {a0_manifest.state.sharing.strategy}")
print(f"  Execution:     Single pass, one agent")
print()

# A1: multi-agent pipeline with state sharing
print("A1 (Adaptive) — Research Pipeline")
print(f"  Agents:        {[n.id for n in manifest.orchestration.graph]}")
deps_summary = {n.id: (n.depends_on or []) for n in manifest.orchestration.graph}
print(f"  Dependencies:  {deps_summary}")
print(f"  State sharing: {manifest.state.sharing.strategy}")
print(f"  Execution:     Topological order, data flows between agents")
print()

print("KEY INSIGHT:")
print("  A0 = one agent does everything.")
print("  A1 = specialized agents collaborate via explicit data handoffs.")
print("  Both are still STATIC — the graph is fixed at design time.")
print("  For DYNAMIC agent spawning, we need A2 (delegation loop).")

## 7. A1 Compliance Checklist

A1 includes all A0 requirements, plus:

- [x] **2+ agents** in the graph
- [x] At least one agent with `depends_on` (DAG edge)
- [x] `state` section present with `state.sharing.strategy`
- [x] Agents declare `share_output` fields
- [x] All agents satisfy the output contract (R17)

### Limitations of A1

Even with multiple agents, A1 is still **fully predetermined**:
- The number of agents is fixed
- The execution order is fixed
- No agent can decide to spawn another agent
- No agent can create new tools

**Next:** Open `A2_delegating.ipynb` to see how the delegation loop engine enables dynamic, manager-driven orchestration.